In [0]:
spark.sql("USE CATALOG MVP_gastos_publicos")
spark.sql("USE SCHEMA gold")

DataFrame[]

In [0]:
%sql
-- Criar tabela Gold com dados tratados e colunas relevantes para analise de negocio

CREATE OR REPLACE TABLE gastos_publicos AS
SELECT
    ID_ANO,
    ID_MES,
    data_referencia,
    ORGAO_DESCRICAO,
    NO_FUNCAO_PT,
    NO_SUBFUNCAO_PT,
    NO_PROGRAMA_PT,
    NO_ACAO,
    DOTACAO_INICIAL,
    DOTACAO_ATUALIZADA,
    DESPESAS_EMPENHADAS,
    DESPESAS_LIQUIDADAS,
    DESPESAS_PAGAS,
    RESTOS_A_PAGAR_PAGOS,
    PAGAMENTOS_TOTAIS,
    Poder_Orgao,
    Primaria_Financeira
FROM MVP_gastos_publicos.silver.gastos_publicos
WHERE PAGAMENTOS_TOTAIS IS NOT NULL  

num_affected_rows,num_inserted_rows


In [0]:
%sql
-- Verificar criacao da tabela Gold
SELECT *
FROM MVP_gastos_publicos.gold.gastos_publicos
LIMIT 10;

ID_ANO,ID_MES,data_referencia,ORGAO_DESCRICAO,NO_FUNCAO_PT,NO_SUBFUNCAO_PT,NO_PROGRAMA_PT,NO_ACAO,DOTACAO_INICIAL,DOTACAO_ATUALIZADA,DESPESAS_EMPENHADAS,DESPESAS_LIQUIDADAS,DESPESAS_PAGAS,RESTOS_A_PAGAR_PAGOS,PAGAMENTOS_TOTAIS,Poder_Orgao,Primaria_Financeira
2024,1,2024-01-01,CAMARA DOS DEPUTADOS,LEGISLATIVA,PROTECAO E BENEFICIOS AO TRABALHADOR,PROGRAMA DE GESTAO E MANUTENCAO DO PODER LEGISLATIVO,"ASSISTENCIA MEDICA E ODONTOLOGICA AOS SERVIDORES CIVIS, EMPR",null,null,null,null,null,225.00,225.00,CD,Prim�ria
2024,1,2024-01-01,CAMARA DOS DEPUTADOS,LEGISLATIVA,ACAO LEGISLATIVA,PROGRAMA DE GESTAO E MANUTENCAO DO PODER LEGISLATIVO,"PROCESSO LEGISLATIVO, FISCALIZACAO E REPRESENTACAO POLITICA",null,null,null,null,null,1024768.40,1024768.40,CD,Prim�ria
2024,1,2024-01-01,CAMARA DOS DEPUTADOS,LEGISLATIVA,ADMINISTRACAO GERAL,PROGRAMA DE GESTAO E MANUTENCAO DO PODER LEGISLATIVO,REFORMA DOS IMOVEIS FUNCIONAIS DESTINADOS A MORADIA DOS DEPU,null,null,null,null,null,89.34,89.34,CD,Prim�ria
2024,1,2024-01-01,CAMARA DOS DEPUTADOS,ENCARGOS ESPECIAIS,OUTROS ENCARGOS ESPECIAIS,OPERACOES ESPECIAIS: OUTROS ENCARGOS ESPECIAIS,BENEFICIOS DE LEGISLACAO ESPECIAL,70000.00,70000.00,70000.00,5648.00,5648.00,null,5648.00,CD,Prim�ria
2024,1,2024-01-01,CAMARA DOS DEPUTADOS,LEGISLATIVA,PROTECAO E BENEFICIOS AO TRABALHADOR,PROGRAMA DE GESTAO E MANUTENCAO DO PODER LEGISLATIVO,"ASSISTENCIA MEDICA E ODONTOLOGICA AOS SERVIDORES CIVIS, EMPR",289605000.00,289605000.00,267344368.53,12821513.51,10585592.23,null,10585592.23,CD,Prim�ria
2024,1,2024-01-01,CAMARA DOS DEPUTADOS,LEGISLATIVA,PROTECAO E BENEFICIOS AO TRABALHADOR,PROGRAMA DE GESTAO E MANUTENCAO DO PODER LEGISLATIVO,"BENEFICIOS OBRIGATORIOS AOS SERVIDORES CIVIS, EMPREGADOS, MI",309946000.00,309946000.00,309946000.00,21783964.18,21770292.43,null,21770292.43,CD,Prim�ria
2024,1,2024-01-01,CAMARA DOS DEPUTADOS,LEGISLATIVA,PROTECAO E BENEFICIOS AO TRABALHADOR,PROGRAMA DE GESTAO E MANUTENCAO DO PODER LEGISLATIVO,"ASSISTENCIA MEDICA E ODONTOLOGICA AOS SERVIDORES CIVIS, EMPR",null,null,null,null,null,11448027.40,11448027.40,CD,Prim�ria
2024,1,2024-01-01,CAMARA DOS DEPUTADOS,LEGISLATIVA,PROTECAO E BENEFICIOS AO TRABALHADOR,PROGRAMA DE GESTAO E MANUTENCAO DO PODER LEGISLATIVO,"BENEFICIOS OBRIGATORIOS AOS SERVIDORES CIVIS, EMPREGADOS, MI",null,null,null,null,null,76372.05,76372.05,CD,Prim�ria
2024,1,2024-01-01,CAMARA DOS DEPUTADOS,LEGISLATIVA,ACAO LEGISLATIVA,PROGRAMA DE GESTAO E MANUTENCAO DO PODER LEGISLATIVO,"PROCESSO LEGISLATIVO, FISCALIZACAO E REPRESENTACAO POLITICA",918073787.76,918073787.76,342668674.82,5864896.26,3655307.70,null,3655307.70,CD,Prim�ria
2024,1,2024-01-01,CAMARA DOS DEPUTADOS,LEGISLATIVA,ADMINISTRACAO GERAL,PROGRAMA DE GESTAO E MANUTENCAO DO PODER LEGISLATIVO,AJUDA DE CUSTO PARA MORADIA OU AUXILIO-MORADIA A AGENTES PUB,9133934.00,9133934.00,9133934.00,439114.91,439114.91,null,439114.91,CD,Prim�ria


In [0]:
%sql
-- Criar tabela agregada para responder os orgaos que mais gastaram e a evolução dos gastos
--- Agregação: SUM dos valores financeiros agrupados por ORGAO + ANO 
CREATE OR REPLACE TABLE gastos_por_orgao_ano AS
SELECT
    ID_ANO,
    ORGAO_DESCRICAO,
    Poder_Orgao,
    COUNT(*) as total_registros,
    SUM(DOTACAO_INICIAL) as dotacao_inicial_total,
    SUM(DOTACAO_ATUALIZADA) as dotacao_atualizada_total,
    SUM(DESPESAS_EMPENHADAS) as despesas_empenhadas_total,
    SUM(DESPESAS_LIQUIDADAS) as despesas_liquidadas_total,
    SUM(DESPESAS_PAGAS) as despesas_pagas_total,
    SUM(PAGAMENTOS_TOTAIS) as pagamentos_totais
FROM MVP_gastos_publicos.silver.gastos_publicos
WHERE PAGAMENTOS_TOTAIS IS NOT NULL
GROUP BY ID_ANO, ORGAO_DESCRICAO, Poder_Orgao
ORDER BY ID_ANO DESC, pagamentos_totais DESC;

num_affected_rows,num_inserted_rows


In [0]:
%sql
-- Criar tabela agregada para responder "Quais areas recebem mais recursos?"

CREATE OR REPLACE TABLE gastos_por_funcao AS
SELECT
    ID_ANO,
    NO_FUNCAO_PT as funcao,
    COUNT(*) as total_registros,
    SUM(DOTACAO_INICIAL) as dotacao_inicial_total,
    SUM(DOTACAO_ATUALIZADA) as dotacao_atualizada_total,
    SUM(DESPESAS_EMPENHADAS) as despesas_empenhadas_total,
    SUM(DESPESAS_LIQUIDADAS) as despesas_liquidadas_total,
    SUM(DESPESAS_PAGAS) as despesas_pagas_total,
    SUM(PAGAMENTOS_TOTAIS) as pagamentos_totais
FROM MVP_gastos_publicos.silver.gastos_publicos
WHERE PAGAMENTOS_TOTAIS IS NOT NULL
GROUP BY ID_ANO, NO_FUNCAO_PT
ORDER BY ID_ANO DESC, pagamentos_totais DESC;

num_affected_rows,num_inserted_rows


In [0]:
%sql
-- Criar uma tabela para serie temporal dos gastos
CREATE OR REPLACE TABLE evolucao_mensal AS
SELECT
    ID_ANO,
    ID_MES,
    data_referencia,
    COUNT(*) as total_registros,
    SUM(DOTACAO_INICIAL) as dotacao_inicial_total,
    SUM(DOTACAO_ATUALIZADA) as dotacao_atualizada_total,
    SUM(DESPESAS_EMPENHADAS) as despesas_empenhadas_total,
    SUM(DESPESAS_LIQUIDADAS) as despesas_liquidadas_total,
    SUM(DESPESAS_PAGAS) as despesas_pagas_total,
    SUM(PAGAMENTOS_TOTAIS) as pagamentos_totais
FROM MVP_gastos_publicos.silver.gastos_publicos
WHERE PAGAMENTOS_TOTAIS IS NOT NULL
GROUP BY ID_ANO, ID_MES, data_referencia
ORDER BY data_referencia;

num_affected_rows,num_inserted_rows


In [0]:
%sql
-- Verificar tabelas Gold foram criadas
SHOW TABLES IN MVP_gastos_publicos.gold;

database,tableName,isTemporary
gold,evolucao_mensal,false
gold,gastos_por_funcao,false
gold,gastos_por_orgao_ano,false
gold,gastos_publicos,false
